# Objective: Demonstrate professional-level data cleaning skills by taking a deliberately messy dataset and systematically transforming it into a clean, analysis-ready dataset. Document every decision. 

### Tech Stack: Python, pandas, numpy, Jupyter Notebook 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder

In [ ]:
df=pd.read_csv('data\googleplaystore.csv')

In [ ]:
df.head()

In [ ]:
#number of rows and columns in the dataset
df.shape

In [ ]:
df.info()

In [ ]:
#renaming column
df.columns= df.columns.str.lower()
df.columns= df.columns.str.replace(" ",'_')
df.columns

In [ ]:
#before any cleaning
before_rows = df.shape[0]
before_duplicates = df.duplicated().sum()
before_nulls = df.isnull().sum().sum()
before_dtypes = df.dtypes

## Null Report


In [ ]:
df.isnull().any()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

### handling each value


In [ ]:
df.columns

In [ ]:
df['type'].unique()

In [ ]:
#only one row contains type=0, remove that entire row
df = df[df['type'] != '0']

In [ ]:
# Fill rating with median
df['rating'] = df['rating'].fillna(df['rating'].median())


The Rating column is a continuous numerical variable. Since app ratings are not perfectly normally distributed and may contain extreme values, the median was chosen instead of the mean because it is less affected by outliers. This preserves the central tendency of the ratings while minimizing the impact of unusually high or low values.

In [ ]:
# Fill android version with mode
df['android_ver'] = df['android_ver'].fillna(df['android_ver'].mode()[0])

The Android Version column specifies the minimum Android version required by an application. Since only three values were missing, replacing them with the most frequently occurring Android version maintains consistency without significantly affecting the dataset.

In [ ]:
# Fill content rating with mode
df['content_rating'] = df['content_rating'].fillna(df['content_rating'].mode()[0])

Content Rating is a categorical variable describing the target audience of an application. Since only one observation was missing, replacing it with the most common category was appropriate and had a negligible effect on the overall dataset.

In [ ]:
# Remove corrupted row
df = df[df['type'] != 0]

# Fill type with mode
df['type'] = df['type'].fillna(df['type'].mode()[0])

The Type column is categorical with two valid values: Free and Paid. As only one value was missing, replacing it with the most frequent category (mode) introduces minimal bias while preserving the record. The invalid value '0' found in this column was identified as part of a corrupted record and the entire row was removed instead of being imputed.

In [ ]:
# Fill current version with Unknown
df['current_ver'] = df['current_ver'].fillna('Unknown')

The Current Version column contains descriptive application version information. Missing values cannot be accurately inferred from other records; therefore, they were replaced with "Unknown". This preserves all observations while clearly indicating that the original information was unavailable.

In [ ]:
#checking again
df.isnull().sum()

## Handling duplicated values

In [ ]:
#check if there are any duplicate rows
df.duplicated().any()

In [ ]:
#count the number of duplicate rows
df.duplicated().sum()

In [ ]:
#removing duplicate values
df.drop_duplicates(inplace=True)


In [ ]:
#duplicates removed
df.duplicated().sum()

In [ ]:
#checking the number of rows again after removing duplicates
df.shape

## Standardisation

In [ ]:
#inspecting each column
for col in df.columns:
    print(f"\n{col}")
    print(df[col].unique()[:20])

In [ ]:
#converting the column 'last_updated' to datetime
df['last_updated']=pd.to_datetime(df['last_updated'])

In [ ]:
print(df['last_updated'].head())
print(" ")
print("data type of the last_updated column:",df['last_updated'].dtype)

last_updated was converted from string format to the datetime data type to enable date-based analysis.

### Standardising the installs column

In [ ]:
df['installs']

In [ ]:
df['installs']=df['installs'].str.replace(",","", regex=False)
df['installs']=df['installs'].str.replace("+", "", regex=False)
df['installs'] = df['installs'].str.replace(" ", "", regex=False)
df['installs'] = df['installs'].astype(int)
df['installs']

installs values were standardized by removing commas and the '+' symbol before converting them to integers.

### Standardising the price column

In [ ]:
df['price'].unique()

In [ ]:
print(df['price'].dtype)

In [ ]:
df['price']=df['price'].str.replace("$","",regex=False)
df['price']=df['price'].astype(float)
df['price'].unique()

price values were standardized by removing the dollar ($) symbol and converting the column to a numeric (float) data type.

### standardising the reviews column

In [ ]:
df['reviews'].dtype

In [ ]:
#converting to numeric column
df['reviews']=pd.to_numeric(df['reviews'])

### Standardising the reviews column

In [ ]:
df['reviews'].dtype

The column was inspected and found to be stored in a numeric format. No additional standardization was required.

### Standardising the size column

In [ ]:
df['size'].dtype

In [ ]:
df['size'].head()

In [ ]:
df['size'].unique()


In [ ]:
import numpy as np

def clean_size(x):
    if x == 'Varies with device':
        return np.nan
    elif 'M' in x:
        return float(x.replace('M', ''))
    elif 'k' in x:
        return float(x.replace('k', '')) / 1024
    else:
        return np.nan

df['size'] = df['size'].apply(clean_size)

reviews was converted from text to an integer data type for numerical analysis.
size values were converted into a single unit (megabytes). Values expressed in kilobytes (k) were converted to megabytes, while "Varies with device" was treated as missing data.

### Standardising the text columns

In [ ]:
text_cols = ['app', 'category', 'type', 'content_rating','genres', 'current_ver', 'android_ver']
for col in text_cols:
    df[col] = df[col].str.strip()

In [ ]:
df['type'].unique()

In [ ]:
df['category'] = df['category'].str.title()
df['content_rating'] = df['content_rating'].str.title()

Leading and trailing whitespaces were removed from all text columns to ensure consistency.

## Outlier Detection

In [ ]:
#identifying numeric columns
numeric_cols = ['rating', 'reviews', 'size', 'installs', 'price']
df[numeric_cols].dtypes

In [ ]:
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower) | (df[col] > upper)]

    print(f"{col}")
    print(f"Number of outliers: {len(outliers)}")
    print("-" * 40)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for col in numeric_cols:
    plt.figure(figsize=(6,2))
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot of {col}')
    plt.savefig(f'images/boxplot of {col}')
    plt.show()

The Interquartile Range (IQR) method was used to identify potential outliers in the numerical columns: Rating, Reviews, Size, Installs, and Price. The IQR method is robust to skewed distributions and effectively highlights observations that fall significantly outside the typical range.

Although several observations were identified as statistical outliers, they were retained because they represent legitimate values rather than data entry errors. For example, highly popular applications naturally have exceptionally high install counts and review counts, while some premium applications have higher prices than the majority of apps. Removing these observations would distort the true characteristics of the Google Play Store dataset.

Only values that were clearly invalid (such as the previously identified corrupted record with an impossible rating) were removed during the data cleaning process

## Data type correction

In [ ]:
for col in df.columns:
    print(col, ':', df[col].dtype)

In [ ]:
#after cleaning the data
after_rows = df.shape[0]
after_duplicates = df.duplicated().sum()
after_nulls = df.isnull().sum().sum()
after_dtypes = df.dtypes

In [ ]:
expected_dtypes = {
    'app':'object',
    'category':'object',
    'rating':'float64',
    'reviews':'int64',
    'size':'float64',
    'installs':'int64',
    'type':'object',
    'price':'float64',
    'content_rating':'object',
    'genres':'object',
    'last_updated':'datetime64[ns]',
    'current_ver':'object',
    'android_ver':'object'
}

In [ ]:
before_dtype_correct = sum(
    str(before_dtypes[col]) == expected_dtypes[col]
    for col in expected_dtypes
)

after_dtype_correct = sum(
    str(after_dtypes[col]) == expected_dtypes[col]
    for col in expected_dtypes
)

In [ ]:
summary = pd.DataFrame({
    "Metric":[
        "Row Count",
        "Total Null Values",
        "Duplicate Rows",
        "Correct Data Types"
    ],
    "Before Cleaning":[
        before_rows,
        before_nulls,
        before_duplicates,
        f"{before_dtype_correct}/13"
    ],
    "After Cleaning":[
        after_rows,
        after_nulls,
        after_duplicates,
        f"{after_dtype_correct}/13"
    ]
})

summary

A comparison of the dataset before and after the data cleaning process demonstrates the improvements made. Missing values were successfully handled using appropriate imputation techniques, duplicate records were removed, inconsistent data formats were standardized, incorrect data types were corrected, and invalid records were eliminated. As a result, the cleaned dataset is complete, consistent, and ready for further analysis or machine learning applications.

## Saving the cleaned data set

In [ ]:
df.to_csv("data/google_playstore_cleaned.csv", index=False)